# Transfer Learning: ResNet as Feature Extractor

---

## Learning Objectives

By the end of this notebook, you will:

- Understand **transfer learning**: reusing pre-trained weights from ImageNet
- Implement **feature extraction**: freeze the backbone and train only a new classification head
- Load a pre-trained ResNet18 and replace the final fully-connected layer
- Freeze and unfreeze model parameters selectively
- Train on a small dataset and compare scratch vs. transfer learning
- Build a complete transfer learning pipeline

## Prerequisites

- **DL100**: Neural network fundamentals
- **DL150**: PyTorch foundations
- **Notebooks 01-03**: CNN basics, training, augmentation

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [What is Transfer Learning?](#2.-What-is-Transfer-Learning?)
3. [Loading Pre-trained ResNet18](#3.-Loading-Pre-trained-ResNet18)
4. [Modifying the Classification Head](#4.-Modifying-the-Classification-Head)
5. [Freezing and Unfreezing Parameters](#5.-Freezing-and-Unfreezing-Parameters)
6. [Data Preparation](#6.-Data-Preparation)
7. [Training: Scratch vs Transfer Learning](#7.-Training:-Scratch-vs-Transfer-Learning)
8. [Results Comparison](#8.-Results-Comparison)
9. [Common Mistakes and Debugging Tips](#9.-Common-Mistakes-and-Debugging-Tips)
10. [Exercises](#10.-Exercises)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import os

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("Setup complete.")

---

## 2. What is Transfer Learning?

**Transfer learning** leverages knowledge from a model trained on a large dataset (e.g., ImageNet with 1.2M images, 1000 classes) and applies it to a new, typically smaller dataset.

### Why It Works

- Early CNN layers learn **general features**: edges, textures, colors
- These features are useful for almost any vision task
- Only the final layers need to be task-specific

### Two Approaches

| Approach | What | When |
|----------|------|------|
| **Feature extraction** | Freeze backbone, train new head only | Small dataset, similar domain |
| **Fine-tuning** | Unfreeze some/all layers, train with small LR | Larger dataset, different domain |

### The Math

A pre-trained CNN can be decomposed as:

$$f(\mathbf{x}) = g_{\text{head}}(h_{\text{backbone}}(\mathbf{x}))$$

- $h_{\text{backbone}}$: feature extractor (conv layers), pre-trained weights $\theta_{\text{bb}}$
- $g_{\text{head}}$: classifier (FC layers), replaced with new weights $\theta_{\text{head}}$

**Feature extraction**: Freeze $\theta_{\text{bb}}$, only optimize $\theta_{\text{head}}$

$$\theta_{\text{head}}^* = \arg\min_{\theta_{\text{head}}} \mathcal{L}(g_{\theta_{\text{head}}}(h_{\theta_{\text{bb}}}(\mathbf{x})), y)$$

---

## 3. Loading Pre-trained ResNet18

ResNet18 is a good starting point for transfer learning:
- Small enough to train quickly
- Powerful enough to extract rich features
- 11.7M parameters, 18 layers deep

In [ ]:
# Load pre-trained ResNet18
try:
    resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    print("Loaded ResNet18 with ImageNet pre-trained weights.")
except Exception:
    # Fallback for older torchvision
    resnet18 = models.resnet18(pretrained=True)
    print("Loaded ResNet18 with pre-trained weights (legacy API).")

# Inspect the architecture
print(f"\nTotal parameters: {sum(p.numel() for p in resnet18.parameters()):,}")
print(f"\nFinal layer (fc): {resnet18.fc}")
print(f"  - Input features:  {resnet18.fc.in_features}")
print(f"  - Output features: {resnet18.fc.out_features} (ImageNet classes)")

In [ ]:
# Explore the layer hierarchy
print("ResNet18 top-level modules:")
for name, module in resnet18.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:12s}: {n_params:>10,} params  ({type(module).__name__})")

---

## 4. Modifying the Classification Head

To use ResNet18 for a different number of classes, we replace the final `fc` layer.

In [ ]:
def create_transfer_model(n_classes, freeze_backbone=True):
    """Create a ResNet18-based transfer learning model.

    Args:
        n_classes: Number of output classes for the new task.
        freeze_backbone: If True, freeze all layers except the new head.

    Returns:
        Modified ResNet18 model.
    """
    # Load pre-trained model
    try:
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    except Exception:
        model = models.resnet18(pretrained=True)

    # Step 1: Freeze backbone if requested
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Step 2: Replace the final FC layer
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, n_classes),
    )
    # Note: the new fc layer has requires_grad=True by default

    return model


# Create model for 10 classes (CIFAR-10)
N_CLASSES = 10
model_transfer = create_transfer_model(N_CLASSES, freeze_backbone=True)

# Count trainable vs frozen parameters
total_params = sum(p.numel() for p in model_transfer.parameters())
trainable_params = sum(p.numel() for p in model_transfer.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:>10,}")
print(f"Trainable parameters: {trainable_params:>10,} ({100*trainable_params/total_params:.1f}%)")
print(f"Frozen parameters:    {frozen_params:>10,} ({100*frozen_params/total_params:.1f}%)")

---

## 5. Freezing and Unfreezing Parameters

Key PyTorch mechanism: `param.requires_grad` controls whether a parameter is updated during backpropagation.

In [ ]:
# Inspect which layers are frozen vs trainable
print("Layer freeze status:")
print(f"{'Layer':<40} {'Shape':<20} {'Trainable':>10}")
print("-" * 72)

for name, param in model_transfer.named_parameters():
    # Only show a subset for readability
    if "layer1.0" in name or "layer4.1" in name or "fc" in name:
        print(f"{name:<40} {str(list(param.shape)):<20} {str(param.requires_grad):>10}")

In [ ]:
# Utility functions for freezing/unfreezing

def freeze_all(model):
    """Freeze all parameters."""
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_all(model):
    """Unfreeze all parameters."""
    for param in model.parameters():
        param.requires_grad = True


def unfreeze_layer(model, layer_name):
    """Unfreeze a specific named layer."""
    for name, param in model.named_parameters():
        if layer_name in name:
            param.requires_grad = True


def count_trainable(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Example: progressive unfreezing
demo_model = create_transfer_model(N_CLASSES, freeze_backbone=True)
print(f"After creation (backbone frozen): {count_trainable(demo_model):,} trainable")

unfreeze_layer(demo_model, "layer4")
print(f"After unfreezing layer4:          {count_trainable(demo_model):,} trainable")

unfreeze_layer(demo_model, "layer3")
print(f"After unfreezing layer3+4:        {count_trainable(demo_model):,} trainable")

unfreeze_all(demo_model)
print(f"After unfreezing all:             {count_trainable(demo_model):,} trainable")

---

## 6. Data Preparation

ResNet18 was trained on ImageNet with specific normalization. We must use the **same normalization** for transfer learning.

ImageNet normalization: $\mu = [0.485, 0.456, 0.406]$, $\sigma = [0.229, 0.224, 0.225]$

In [ ]:
# ImageNet normalization values
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ResNet expects 224x224 input, but we can use smaller sizes
# For CIFAR-10 (32x32), we resize to a reasonable size
RESIZE_DIM = 64  # compromise between speed and quality

# Transforms for transfer learning
train_transform = transforms.Compose([
    transforms.Resize(RESIZE_DIM),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(RESIZE_DIM),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

data_dir = os.path.join("..", "..", "data")

try:
    train_dataset = datasets.CIFAR10(data_dir, train=True, download=True, transform=train_transform)
    test_dataset = datasets.CIFAR10(data_dir, train=False, download=True, transform=eval_transform)
    DATASET_NAME = "CIFAR-10"
    class_names = train_dataset.classes

    # For scratch model: transforms without ImageNet normalization
    train_transform_scratch = transforms.Compose([
        transforms.Resize(RESIZE_DIM),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
    ])
    eval_transform_scratch = transforms.Compose([
        transforms.Resize(RESIZE_DIM),
        transforms.ToTensor(),
        transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
    ])
    train_dataset_scratch = datasets.CIFAR10(data_dir, train=True, download=True, transform=train_transform_scratch)
    test_dataset_scratch = datasets.CIFAR10(data_dir, train=False, download=True, transform=eval_transform_scratch)

except Exception as e:
    print(f"CIFAR-10 download failed: {e}")
    print("Falling back to MNIST (converted to 3 channels for ResNet).")

    # MNIST needs to be converted to 3 channels for ResNet
    train_transform = transforms.Compose([
        transforms.Resize(RESIZE_DIM),
        transforms.Grayscale(num_output_channels=3),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize(RESIZE_DIM),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    train_dataset = datasets.MNIST(data_dir, train=True, download=True, transform=train_transform)
    test_dataset = datasets.MNIST(data_dir, train=False, download=True, transform=eval_transform)
    DATASET_NAME = "MNIST"
    class_names = [str(i) for i in range(10)]

    train_transform_scratch = train_transform
    eval_transform_scratch = eval_transform
    train_dataset_scratch = datasets.MNIST(data_dir, train=True, download=True, transform=train_transform_scratch)
    test_dataset_scratch = datasets.MNIST(data_dir, train=False, download=True, transform=eval_transform_scratch)

print(f"Dataset: {DATASET_NAME}")
print(f"Classes: {class_names}")

In [ ]:
# Use a SMALL subset to demonstrate the power of transfer learning
# Transfer learning shines when data is limited
TRAIN_SUBSET = 2000  # small dataset to show transfer learning advantage
BATCH_SIZE = 64

set_global_seed(42)
train_indices = torch.randperm(len(train_dataset))[:TRAIN_SUBSET].tolist()

train_sub = Subset(train_dataset, train_indices)
train_sub_scratch = Subset(train_dataset_scratch, train_indices)

train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
train_loader_scratch = DataLoader(train_sub_scratch, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=0)
test_loader_scratch = DataLoader(test_dataset_scratch, batch_size=BATCH_SIZE, num_workers=0)

print(f"Training subset: {TRAIN_SUBSET} samples ({TRAIN_SUBSET / len(train_dataset) * 100:.1f}% of full set)")
print(f"Test set: {len(test_dataset)} samples")
print(f"Batch size: {BATCH_SIZE}")

---

## 7. Training: Scratch vs Transfer Learning

In [ ]:
def train_model(model, train_loader, test_loader, epochs=15, lr=1e-3, label=""):
    """Train model and return history."""
    loss_fn = nn.CrossEntropyLoss()
    # Only optimize parameters with requires_grad=True
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        losses, correct, total = [], 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(X_b)
            loss = loss_fn(logits, y_b)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            correct += (logits.argmax(-1) == y_b).sum().item()
            total += y_b.size(0)

        history["train_loss"].append(np.mean(losses))
        history["train_acc"].append(correct / total)

        # Test
        model.eval()
        losses, correct, total = [], 0, 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits = model(X_b)
                losses.append(loss_fn(logits, y_b).item())
                correct += (logits.argmax(-1) == y_b).sum().item()
                total += y_b.size(0)

        history["test_loss"].append(np.mean(losses))
        history["test_acc"].append(correct / total)

        if epoch % 3 == 0 or epoch == 1:
            print(f"  [{label}] Epoch {epoch:3d}/{epochs} | "
                  f"Train Acc: {history['train_acc'][-1]:.4f} | "
                  f"Test Acc: {history['test_acc'][-1]:.4f}")

    return history

In [ ]:
# Model 1: ResNet18 from scratch (random init, all params trainable)
EPOCHS = 15

print("=" * 60)
print("Model 1: ResNet18 from SCRATCH (random initialization)")
print("=" * 60)

set_global_seed(42)
model_scratch = models.resnet18(weights=None)  # no pre-trained weights
model_scratch.fc = nn.Sequential(
    nn.Linear(model_scratch.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, N_CLASSES),
)
model_scratch = model_scratch.to(device)

n_trainable_scratch = sum(p.numel() for p in model_scratch.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_trainable_scratch:,}")

history_scratch = train_model(
    model_scratch, train_loader_scratch, test_loader_scratch,
    epochs=EPOCHS, lr=1e-3, label="Scratch"
)

In [ ]:
# Model 2: Transfer learning (pre-trained backbone, frozen, new head)
print("\n" + "=" * 60)
print("Model 2: Transfer Learning (frozen backbone, new head)")
print("=" * 60)

set_global_seed(42)
model_transfer = create_transfer_model(N_CLASSES, freeze_backbone=True).to(device)

n_trainable_transfer = sum(p.numel() for p in model_transfer.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_trainable_transfer:,} (only the new head)")

history_transfer = train_model(
    model_transfer, train_loader, test_loader,
    epochs=EPOCHS, lr=1e-3, label="Transfer"
)

---

## 8. Results Comparison

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, EPOCHS + 1)

# Loss
ax1.plot(ep, history_scratch["train_loss"], label="Scratch - Train", linestyle="-", color="C0")
ax1.plot(ep, history_scratch["test_loss"], label="Scratch - Test", linestyle="--", color="C0")
ax1.plot(ep, history_transfer["train_loss"], label="Transfer - Train", linestyle="-", color="C1")
ax1.plot(ep, history_transfer["test_loss"], label="Transfer - Test", linestyle="--", color="C1")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss: Scratch vs Transfer Learning")
ax1.legend(); ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(ep, history_scratch["train_acc"], label="Scratch - Train", linestyle="-", color="C0")
ax2.plot(ep, history_scratch["test_acc"], label="Scratch - Test", linestyle="--", color="C0")
ax2.plot(ep, history_transfer["train_acc"], label="Transfer - Train", linestyle="-", color="C1")
ax2.plot(ep, history_transfer["test_acc"], label="Transfer - Test", linestyle="--", color="C1")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy: Scratch vs Transfer Learning")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"Training set size: {TRAIN_SUBSET} samples")
print(f"{'Model':<30} {'Trainable Params':<18} {'Best Test Acc':<15} {'Final Test Acc':<15}")
print("-" * 78)
print(f"{'Scratch (ResNet18)':<30} {n_trainable_scratch:<18,} "
      f"{max(history_scratch['test_acc']):<15.4f} {history_scratch['test_acc'][-1]:<15.4f}")
print(f"{'Transfer (frozen backbone)':<30} {n_trainable_transfer:<18,} "
      f"{max(history_transfer['test_acc']):<15.4f} {history_transfer['test_acc'][-1]:<15.4f}")

improvement = max(history_transfer['test_acc']) - max(history_scratch['test_acc'])
print(f"\nTransfer learning improvement: {improvement:+.4f} ({improvement*100:+.2f} percentage points)")
print(f"Trainable params reduction: {(1 - n_trainable_transfer/n_trainable_scratch)*100:.1f}% fewer params to train")

---

## 9. Common Mistakes and Debugging Tips

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Wrong normalization (not ImageNet stats) | Poor transfer performance | Use `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]` |
| Forgetting to freeze backbone | Training is slow, overfits on small data | Set `requires_grad = False` for backbone params |
| Passing frozen params to optimizer | Optimizer warning or error | Use `filter(lambda p: p.requires_grad, model.parameters())` |
| Not replacing the final FC layer | 1000-class output instead of N | Replace `model.fc` with a new `nn.Linear` |
| Using 1-channel images with ResNet | Shape error (expects 3 channels) | Convert to 3 channels with `Grayscale(num_output_channels=3)` |
| Learning rate too high for fine-tuning | Destroys pre-trained features | Use small LR ($10^{-4}$ to $10^{-5}$) when unfreezing backbone |
| Not calling `model.eval()` during inference | BatchNorm/Dropout behave differently | Always toggle `train()`/`eval()` |

---

## 10. Exercises

### Exercise 1: Try a Different Pre-trained Model

Replace ResNet18 with another pre-trained model (e.g., `models.resnet34`, `models.mobilenet_v2`, or `models.efficientnet_b0`). Compare transfer learning results.

Hint: Different models have different final layer names:
- ResNet: `model.fc`
- MobileNet: `model.classifier[-1]`
- EfficientNet: `model.classifier[-1]`

In [ ]:
# ===== EXERCISE 1: Your code here =====
# try:
#     model_alt = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
#     for param in model_alt.parameters():
#         param.requires_grad = False
#     # Replace classifier
#     in_features = model_alt.classifier[-1].in_features
#     model_alt.classifier[-1] = nn.Linear(in_features, N_CLASSES)
#     model_alt = model_alt.to(device)
#     ...
# except Exception as e:
#     print(f"Error: {e}")
pass

### Exercise 2: Progressive Unfreezing

Implement a two-phase training approach:

1. **Phase 1**: Freeze backbone, train head only for 5 epochs (lr=1e-3)
2. **Phase 2**: Unfreeze layer4, train for 10 more epochs (lr=1e-4)

Compare with the fully-frozen approach.

In [ ]:
# ===== EXERCISE 2: Your code here =====
# Phase 1: train head only
# Phase 2: unfreeze layer4, lower LR
pass

### Exercise 1 -- Solution

In [ ]:
models_to_try = {}

# ResNet18 (already done)
models_to_try["ResNet18"] = history_transfer

# Try MobileNetV2
try:
    set_global_seed(42)
    try:
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    except Exception:
        mobilenet = models.mobilenet_v2(pretrained=True)

    for param in mobilenet.parameters():
        param.requires_grad = False

    in_feat = mobilenet.classifier[-1].in_features
    mobilenet.classifier[-1] = nn.Sequential(
        nn.Linear(in_feat, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, N_CLASSES),
    )
    mobilenet = mobilenet.to(device)

    n_trainable_mob = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
    print(f"MobileNetV2 trainable params: {n_trainable_mob:,}")

    history_mobilenet = train_model(
        mobilenet, train_loader, test_loader,
        epochs=EPOCHS, lr=1e-3, label="MobileNet"
    )
    models_to_try["MobileNetV2"] = history_mobilenet
except Exception as e:
    print(f"MobileNetV2 failed: {e}")

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
ep = range(1, EPOCHS + 1)
for name, h in models_to_try.items():
    ax.plot(ep, h["test_acc"], label=name, linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Test Accuracy")
ax.set_title("Transfer Learning: Model Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for name, h in models_to_try.items():
    print(f"{name:20s}: Best Test Acc = {max(h['test_acc']):.4f}")

### Exercise 2 -- Solution

In [ ]:
# Progressive unfreezing
set_global_seed(42)
model_progressive = create_transfer_model(N_CLASSES, freeze_backbone=True).to(device)

# Phase 1: Train head only (5 epochs, lr=1e-3)
print("Phase 1: Training head only")
print(f"  Trainable params: {count_trainable(model_progressive):,}")

loss_fn = nn.CrossEntropyLoss()
optimizer_p1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_progressive.parameters()), lr=1e-3
)

history_prog = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(1, 6):
    model_progressive.train()
    losses, correct, total = [], 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_p1.zero_grad()
        logits = model_progressive(X_b)
        loss = loss_fn(logits, y_b)
        loss.backward()
        optimizer_p1.step()
        losses.append(loss.item())
        correct += (logits.argmax(-1) == y_b).sum().item()
        total += y_b.size(0)
    history_prog["train_loss"].append(np.mean(losses))
    history_prog["train_acc"].append(correct / total)

    model_progressive.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model_progressive(X_b)
            losses.append(loss_fn(logits, y_b).item())
            correct += (logits.argmax(-1) == y_b).sum().item()
            total += y_b.size(0)
    history_prog["test_loss"].append(np.mean(losses))
    history_prog["test_acc"].append(correct / total)
    print(f"  Epoch {epoch}: Train Acc={history_prog['train_acc'][-1]:.4f}, Test Acc={history_prog['test_acc'][-1]:.4f}")

# Phase 2: Unfreeze layer4, train with lower LR
print("\nPhase 2: Unfreezing layer4")
unfreeze_layer(model_progressive, "layer4")
print(f"  Trainable params: {count_trainable(model_progressive):,}")

optimizer_p2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_progressive.parameters()), lr=1e-4
)

for epoch in range(6, 16):
    model_progressive.train()
    losses, correct, total = [], 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_p2.zero_grad()
        logits = model_progressive(X_b)
        loss = loss_fn(logits, y_b)
        loss.backward()
        optimizer_p2.step()
        losses.append(loss.item())
        correct += (logits.argmax(-1) == y_b).sum().item()
        total += y_b.size(0)
    history_prog["train_loss"].append(np.mean(losses))
    history_prog["train_acc"].append(correct / total)

    model_progressive.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model_progressive(X_b)
            losses.append(loss_fn(logits, y_b).item())
            correct += (logits.argmax(-1) == y_b).sum().item()
            total += y_b.size(0)
    history_prog["test_loss"].append(np.mean(losses))
    history_prog["test_acc"].append(correct / total)

    if epoch % 3 == 0:
        print(f"  Epoch {epoch}: Train Acc={history_prog['train_acc'][-1]:.4f}, Test Acc={history_prog['test_acc'][-1]:.4f}")

# Compare
print(f"\nProgressive unfreezing best test acc: {max(history_prog['test_acc']):.4f}")
print(f"Fully frozen best test acc:            {max(history_transfer['test_acc']):.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, 16), history_prog["test_acc"], label="Progressive Unfreezing", linewidth=2)
ax.plot(range(1, EPOCHS + 1), history_transfer["test_acc"], label="Fully Frozen", linewidth=2)
ax.axvline(5.5, color="gray", linestyle="--", alpha=0.5, label="Unfreeze layer4")
ax.set_xlabel("Epoch")
ax.set_ylabel("Test Accuracy")
ax.set_title("Progressive Unfreezing vs Fully Frozen")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

**Next notebook:** [05 -- Fine-Tuning Best Practices](05_FineTuning_Best_Practices.ipynb)